<a href="https://colab.research.google.com/github/teguhsmlnna666/BigData26_B_2411532005_TeguhEsaMaulanna/blob/main/Praktikum3/BD_T03_2411532005_TeguhEsaMaulanna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tugas Praktikum

Langkah ini untuk menyiapkan lingkungan kerja Google Colab, memeriksa versi library yang digunakan, menghubungkan Google Drive, membuat direktori penyimpanan data, serta menyiapkan fungsi pengukuran waktu, penggunaan memori, dan pencatatan data lineage sebagai asal-usul data selama proses praktikum.

In [1]:
import sys, time, os, json, sqlite3, gc
import pandas as pd, numpy as np, psutil

for nama in ["pandas", "numpy", "pyarrow", "requests", "sklearn", "sqlalchemy"]:
    try:
        mod = __import__(nama)
        print(f"{nama:11s} : {mod.__version__}")
    except ImportError:
        print(f"{nama:11s} : BELUM TERPASANG")

from google.colab import drive
drive.mount("/content/drive")

DIR_MENTAH = "/content/lapisan_mentah"
DIR_KURASI = "/content/lapisan_terkurasi"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum3/tugas"

for d in (DIR_MENTAH, DIR_KURASI, DIR_SIMPAN):
    os.makedirs(d, exist_ok=True)

# Dipakai kembali dari Praktikum 1 / K-3
proses = psutil.Process(os.getpid())
catatan = []

def rss_mb():
    return proses.memory_info().rss / 1024**2

def ukur(label, fungsi):
    m0, t0 = rss_mb(), time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({
        "langkah": label,
        "detik": round(detik, 2),
        "delta_rss_mb": round(rss_mb() - m0, 1)
    })
    print(f"[{label}] {detik:.2f} s")
    return hasil

# Baru di Praktikum 3: catatan asal-usul data (lineage)
lineage = []

def catat_sumber(nama, asal, jumlah_baris, keterangan=""):
    lineage.append({
        "sumber": nama,
        "asal": asal,
        "diambil_pada": pd.Timestamp.now("UTC").isoformat(),
        "jumlah_baris": jumlah_baris,
        "keterangan": keterangan,
    })
    print(f"[lineage] {nama}: {jumlah_baris:,} baris")

pandas      : 2.2.3
numpy       : 2.1.3
pyarrow     : 23.0.1
requests    : 2.32.4
sklearn     : 1.6.1
sqlalchemy  : 2.0.54
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Langkah ini untuk mengunduh data perjalanan dan zona taksi secara aman dengan mekanisme retry, validasi file, dan penulisan atomik, kemudian membaca kolom yang diperlukan serta mencatat sumber dan jumlah baris data melalui data lineage. Dataset perjalanan yang digunakan yaitu bulan Januari dan Desember 2023 untuk melihat perbandingan hasil pengolahan data pada dua periode yang berbeda.

In [3]:
import requests, shutil, os, time

def unduh_aman(url, tujuan, percobaan=3, jeda_awal=2):
    """Unduh dengan retry + penulisan atomik. Melewati unduhan bila file sudah ada."""
    if os.path.exists(tujuan) and os.path.getsize(tujuan) > 0:
        print("cache ditemukan:", os.path.basename(tujuan))
        return tujuan

    sementara = tujuan + ".part"

    # Menambahkan identitas browser agar tidak diblokir server
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

    for i in range(percobaan):
        try:
            with requests.get(url, stream=True, timeout=60, headers=headers) as r:
                r.raise_for_status()
                with open(sementara, "wb") as f:
                    shutil.copyfileobj(r.raw, f)

            # Validasi jika file sangat kecil
            if os.path.getsize(sementara) < 1000:
                raise IOError(
                    f"berkas terlalu kecil atau kosong "
                    f"(ukuran: {os.path.getsize(sementara)} bytes)"
                )

            os.replace(sementara, tujuan)
            return tujuan

        except Exception as e:
            jeda = jeda_awal * (2 ** i)
            print(f"percobaan {i+1} gagal ({e}); menunggu {jeda}s")
            time.sleep(jeda)

    raise RuntimeError(
        f"gagal mengunduh setelah {percobaan} percobaan: {url}"
    )

URL_TRIP_JAN = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
URL_TRIP_DES = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-12.parquet"
URL_ZONA = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

PATH_TRIP_JAN = os.path.join(
    DIR_MENTAH, "yellow_tripdata_2023-01.parquet"
)

PATH_TRIP_DES = os.path.join(
    DIR_MENTAH, "yellow_tripdata_2023-12.parquet"
)

PATH_ZONA = os.path.join(
    DIR_MENTAH, "taxi_zone_lookup.csv"
)

ukur(
    "unduh trip Januari",
    lambda: unduh_aman(URL_TRIP_JAN, PATH_TRIP_JAN)
)

ukur(
    "unduh trip Desember",
    lambda: unduh_aman(URL_TRIP_DES, PATH_TRIP_DES)
)

ukur(
    "unduh zona",
    lambda: unduh_aman(URL_ZONA, PATH_ZONA)
)

for pth in (PATH_TRIP_JAN, PATH_TRIP_DES, PATH_ZONA):
    print(
        os.path.basename(pth),
        "->",
        round(os.path.getsize(pth) / 1024**2, 2),
        "MB"
    )

KOLOM = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance",
    "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "tip_amount", "total_amount"
]

trip_jan = ukur(
    "baca trip Januari",
    lambda: pd.read_parquet(PATH_TRIP_JAN, columns=KOLOM)
)

trip_des = ukur(
    "baca trip Desember",
    lambda: pd.read_parquet(PATH_TRIP_DES, columns=KOLOM)
)

# taxi_zone_lookup.csv dari server TLC terbaca sebagai gzip
zona = pd.read_csv(PATH_ZONA, compression="gzip")

catat_sumber(
    "trip_januari",
    URL_TRIP_JAN,
    len(trip_jan),
    "Parquet bulanan TLC Januari 2023"
)

catat_sumber(
    "trip_desember",
    URL_TRIP_DES,
    len(trip_des),
    "Parquet bulanan TLC Desember 2023"
)

catat_sumber(
    "zona",
    URL_ZONA,
    len(zona),
    "tabel dimensi taxi zone"
)

print("\nUkuran dataset:")
print("Januari  :", f"{len(trip_jan):,}", "baris")
print("Desember :", f"{len(trip_des):,}", "baris")
print("Zona     :", f"{len(zona):,}", "baris")

display(zona.head())

cache ditemukan: yellow_tripdata_2023-01.parquet
[unduh trip Januari] 0.00 s
cache ditemukan: yellow_tripdata_2023-12.parquet
[unduh trip Desember] 0.00 s
cache ditemukan: taxi_zone_lookup.csv
[unduh zona] 0.00 s
yellow_tripdata_2023-01.parquet -> 45.46 MB
yellow_tripdata_2023-12.parquet -> 54.17 MB
taxi_zone_lookup.csv -> 0.0 MB
[baca trip Januari] 1.24 s
[baca trip Desember] 0.68 s
[lineage] trip_januari: 3,066,766 baris
[lineage] trip_desember: 3,376,567 baris
[lineage] zona: 265 baris

Ukuran dataset:
Januari  : 3,066,766 baris
Desember : 3,376,567 baris
Zona     : 265 baris


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


Langkah ini untuk mengambil data cuaca dan hari libur dari API publik sebagai sumber data tambahan. Data cuaca diperoleh dari Open-Meteo berdasarkan periode Januari dan Desember 2023, sedangkan data hari libur diperoleh dari Nager.Date API, kemudian data dipersiapkan agar dapat diintegrasikan dengan dataset perjalanan berdasarkan waktu dan tanggal.

In [11]:
API_CUACA = "https://archive-api.open-meteo.com/v1/archive"
API_LIBUR = "https://date.nager.at/api/v3/PublicHolidays/2023/US"

def ambil_cuaca(mulai, selesai, lat=40.7128, lon=-74.0060, percobaan=3):
    parameter = {
        "latitude": lat, "longitude": lon,
        "start_date": mulai, "end_date": selesai,
        "hourly": "temperature_2m,precipitation",
        "timezone": "America/New_York",
    }

    for i in range(percobaan):
        r = requests.get(API_CUACA, params=parameter, timeout=60)

        if r.status_code == 200:
            data = r.json()

            if "hourly" not in data:
                raise ValueError("kunci 'hourly' tidak ada pada respons")

            return pd.DataFrame(data["hourly"])

        if r.status_code in (429, 500, 502, 503):
            time.sleep(2**i)
            continue

        r.raise_for_status()

    raise RuntimeError("API cuaca tidak merespons dengan benar")


# Cuaca Januari 2023
cuaca_jan = ukur(
    "ambil cuaca Januari",
    lambda: ambil_cuaca("2023-01-01", "2023-01-31")
)

# Cuaca Desember 2023
cuaca_des = ukur(
    "ambil cuaca Desember",
    lambda: ambil_cuaca("2023-12-01", "2023-12-31")
)

for df in (cuaca_jan, cuaca_des):
    df["time"] = pd.to_datetime(df["time"])
    df.rename(
        columns={
            "time": "jam_mulai",
            "temperature_2m": "suhu_c",
            "precipitation": "hujan_mm"
        },
        inplace=True
    )

# Gabungkan cuaca Januari dan Desember
cuaca = pd.concat(
    [cuaca_jan, cuaca_des],
    ignore_index=True
)

cuaca = cuaca.drop_duplicates(
    subset=["jam_mulai"]
).sort_values("jam_mulai").reset_index(drop=True)


catat_sumber(
    "cuaca_januari",
    API_CUACA,
    len(cuaca_jan),
    "Open-Meteo hourly archive Januari 2023"
)

catat_sumber(
    "cuaca_desember",
    API_CUACA,
    len(cuaca_des),
    "Open-Meteo hourly archive Desember 2023"
)

def ambil_libur(url=API_LIBUR, percobaan=3):
    for i in range(percobaan):
        r = requests.get(url, timeout=60)

        if r.status_code == 200:
            data = r.json()

            if not isinstance(data, list):
                raise ValueError(
                    "format respons API hari libur bukan list"
                )

            return pd.DataFrame(data)

        if r.status_code in (429, 500, 502, 503):
            time.sleep(2**i)
            continue

        r.raise_for_status()

    raise RuntimeError(
        "API hari libur tidak merespons dengan benar"
    )


libur = ukur(
    "ambil hari libur nasional",
    lambda: ambil_libur()
)

# Ambil kolom yang diperlukan untuk proses join
libur = libur[
    ["date", "localName", "name", "global", "types"]
].copy()

libur["date"] = pd.to_datetime(libur["date"])

libur = libur.rename(
    columns={
        "date": "tanggal",
        "localName": "nama_libur_lokal",
        "name": "nama_libur",
        "global": "libur_nasional",
        "types": "jenis_libur"
    }
)

libur = (
    libur
    .sort_values(["tanggal", "nama_libur"])
    .drop_duplicates(
        subset=["tanggal"],
        keep="first"
    )
    .reset_index(drop=True)
)

catat_sumber(
    "hari_libur",
    API_LIBUR,
    len(libur),
    "Nager.Date public holidays Amerika Serikat 2023"
)

print("\nJumlah data:")
print("Cuaca Januari  :", f"{len(cuaca_jan):,}", "baris")
print("Cuaca Desember :", f"{len(cuaca_des):,}", "baris")
print("Cuaca gabungan :", f"{len(cuaca):,}", "baris")
print("Hari libur     :", f"{len(libur):,}", "baris")

print("\nTanggal hari libur unik?",
      libur["tanggal"].is_unique)

print("\nContoh data cuaca:")
display(cuaca.head())

print("\nContoh data hari libur:")
display(
    libur[
        [
            "tanggal",
            "nama_libur_lokal",
            "nama_libur",
            "libur_nasional"
        ]
    ].head()
)

[ambil cuaca Januari] 0.54 s
[ambil cuaca Desember] 0.49 s
[lineage] cuaca_januari: 744 baris
[lineage] cuaca_desember: 744 baris
[ambil hari libur nasional] 0.21 s
[lineage] hari_libur: 14 baris

Jumlah data:
Cuaca Januari  : 744 baris
Cuaca Desember : 744 baris
Cuaca gabungan : 1,488 baris
Hari libur     : 14 baris

Tanggal hari libur unik? True

Contoh data cuaca:


,jam_mulai,suhu_c,hujan_mm
0,2023-01-01 00:00:00,10.9,1.0
1,2023-01-01 01:00:00,10.6,1.0
2,2023-01-01 02:00:00,10.6,0.1
3,2023-01-01 03:00:00,10.5,0.0
4,2023-01-01 04:00:00,9.8,0.0



Contoh data hari libur:


,tanggal,nama_libur_lokal,nama_libur,libur_nasional
0,2023-01-02,New Year's Day,New Year's Day,True
1,2023-01-16,"Martin Luther King, Jr. Day","Martin Luther King, Jr. Day",True
2,2023-02-12,Lincoln's Birthday,Lincoln's Birthday,False
3,2023-02-20,Washington's Birthday,Presidents Day,True
4,2023-04-07,Good Friday,Good Friday,False


Langkah ini untuk membuat basis data operasional menggunakan SQLite yang berisi tabel referensi jenis pembayaran dan sebagian data perjalanan. Proses ini digunakan untuk menunjukkan integrasi data dari basis data relasional, melakukan query agregasi, membaca data secara bertahap, serta mencatat tabel referensi sebagai salah satu sumber data yang digunakan dalam pipeline.

In [6]:
from sqlalchemy import create_engine

PATH_DB = os.path.join(DIR_MENTAH, "operasional.db")

engine = create_engine(f"sqlite:///{PATH_DB}")

tarif_referensi = pd.DataFrame({
    "payment_type": [1, 2, 3, 4, 5, 6],
    "nama_pembayaran": [
        "Kartu kredit",
        "Tunai",
        "Gratis",
        "Sengketa",
        "Tidak diketahui",
        "Perjalanan batal"
    ],
    "kena_biaya_admin": [1, 0, 0, 0, 0, 0],
})

tarif_referensi.to_sql(
    "tarif_referensi",
    engine,
    if_exists="replace",
    index=False
)

# Tabel transaksi tiruan untuk latihan pembacaan bertahap
trip_jan.head(300_000).to_sql(
    "trip_operasional",
    engine,
    if_exists="replace",
    index=False,
    chunksize=50_000
)

print(
    "Tabel dibuat:",
    pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table'",
        engine
    )["name"].tolist()
)

SQL = """
SELECT payment_type, COUNT(*) AS jumlah, AVG(total_amount) AS rata_total
FROM trip_operasional
WHERE total_amount > 0
GROUP BY payment_type
ORDER BY jumlah DESC
"""

ringkas_db = pd.read_sql(SQL, engine)

print(ringkas_db)

# Pembacaan bertahap: agregasi tanpa memuat seluruh tabel ke memori
total_baris = 0

for bagian in pd.read_sql(
    "SELECT trip_distance FROM trip_operasional",
    engine,
    chunksize=50_000
):
    total_baris += len(bagian)

print(
    "dibaca bertahap:",
    f"{total_baris:,}",
    "baris"
)

tarif_referensi = pd.read_sql(
    "SELECT * FROM tarif_referensi",
    engine
)

catat_sumber(
    "tarif_referensi",
    PATH_DB,
    len(tarif_referensi),
    "tabel dimensi dari basis data operasional"
)

Tabel dibuat: ['tarif_referensi', 'trip_operasional']
   payment_type  jumlah  rata_total
0             1  226967   31.339172
1             2   66504   25.218836
2             4    2082   26.611114
3             3    1553   22.089691
dibaca bertahap: 300,000 baris
[lineage] tarif_referensi: 6 baris


Langkah ini untuk melakukan profiling dan pemeriksaan kualitas data pada dataset Januari dan Desember 2023. Pemeriksaan dilakukan terhadap kelengkapan, duplikasi, validitas jarak dan nilai transaksi, keberadaan zona, durasi perjalanan, konsistensi nilai transaksi, serta kesesuaian waktu perjalanan dengan periode data, kemudian hasilnya dibandingkan melalui laporan kualitas data.

In [7]:
def profil_kolom(df):
    baris = []

    for kol in df.columns:
        s = df[kol]

        baris.append({
            "kolom": kol,
            "tipe": str(s.dtype),
            "persen_hilang": round(100 * s.isna().mean(), 3),
            "nilai_unik": s.nunique(dropna=True),
            "contoh": s.dropna().iloc[0] if s.notna().any() else None,
        })

    return pd.DataFrame(baris)

trip_jan["durasi_menit"] = (
    (
        trip_jan["tpep_dropoff_datetime"]
        - trip_jan["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60
)

profil_jan = profil_kolom(trip_jan)

print("Profil dataset Januari 2023:")
display(profil_jan)

trip_des["durasi_menit"] = (
    (
        trip_des["tpep_dropoff_datetime"]
        - trip_des["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60
)

profil_des = profil_kolom(trip_des)

print("Profil dataset Desember 2023:")
display(profil_des)

zona_sah = set(zona["LocationID"])

def buat_laporan_kualitas(trip, bulan, awal, akhir):
    aturan = {
        "kelengkapan: passenger_count ada":
            trip["passenger_count"].notna(),

        "keunikan: baris tidak duplikat":
            ~trip.duplicated(),

        "validitas: jarak 0-100 mil":
            trip["trip_distance"].between(0.01, 100),

        "validitas: total_amount > 0":
            trip["total_amount"] > 0,

        "validitas: PULocationID dikenal":
            trip["PULocationID"].isin(zona_sah),

        "konsistensi: durasi 1-180 menit":
            trip["durasi_menit"].between(1, 180),

        "konsistensi: total >= fare":
            trip["total_amount"] >= trip["fare_amount"],

        "ketepatan waktu dalam bulan":
            (
                (trip["tpep_pickup_datetime"] >= awal)
                & (trip["tpep_pickup_datetime"] < akhir)
            ),
    }

    laporan_bulan = pd.DataFrame([
        {
            "bulan": bulan,
            "aturan": nama,
            "lulus": int(mask.sum()),
            "gagal": int((~mask).sum()),
            "persen_gagal": round(100 * (~mask).mean(), 3),
        }
        for nama, mask in aturan.items()
    ])

    return laporan_bulan.sort_values(
        "persen_gagal",
        ascending=False
    )

# Januari 2023
laporan_jan = buat_laporan_kualitas(
    trip_jan,
    "2023-01",
    pd.Timestamp("2023-01-01"),
    pd.Timestamp("2023-02-01")
)

# Desember 2023
laporan_des = buat_laporan_kualitas(
    trip_des,
    "2023-12",
    pd.Timestamp("2023-12-01"),
    pd.Timestamp("2024-01-01")
)

# Gabungkan laporan dua bulan
laporan = pd.concat(
    [laporan_jan, laporan_des],
    ignore_index=True
)

print("Laporan kualitas Januari dan Desember:")
display(laporan)

Profil dataset Januari 2023:


,kolom,tipe,persen_hilang,nilai_unik,contoh
0,tpep_pickup_datetime,datetime64[us],0.000,1610975,2023-01-01 00:32:10
1,tpep_dropoff_datetime,datetime64[us],0.000,1611319,2023-01-01 00:40:36
2,passenger_count,float64,2.339,10,1.0
3,trip_distance,float64,0.000,4387,0.97
4,PULocationID,int64,0.000,257,161
5,DOLocationID,int64,0.000,261,141
6,payment_type,int64,0.000,5,2
7,fare_amount,float64,0.000,6873,9.3
8,tip_amount,float64,0.000,4036,0.0
9,total_amount,float64,0.000,15871,14.3


Profil dataset Desember 2023:


,kolom,tipe,persen_hilang,nilai_unik,contoh
0,tpep_pickup_datetime,datetime64[us],0.000,1666104,2023-12-01 00:06:06
1,tpep_dropoff_datetime,datetime64[us],0.000,1665497,2023-12-01 00:15:47
2,passenger_count,float64,5.331,10,0.0
3,trip_distance,float64,0.000,4539,1.1
4,PULocationID,int32,0.000,259,230
5,DOLocationID,int32,0.000,260,48
6,payment_type,int64,0.000,5,1
7,fare_amount,float64,0.000,9896,10.0
8,tip_amount,float64,0.000,4428,1.5
9,total_amount,float64,0.000,19681,16.5


Laporan kualitas Januari dan Desember:


,bulan,aturan,lulus,gagal,persen_gagal
0,2023-01,kelengkapan: passenger_count ada,2995023,71743,2.339
1,2023-01,validitas: jarak 0-100 mil,3020816,45950,1.498
2,2023-01,konsistensi: durasi 1-180 menit,3030383,36383,1.186
3,2023-01,validitas: total_amount > 0,3040994,25772,0.840
4,2023-01,konsistensi: total >= fare,3041743,25023,0.816
5,2023-01,ketepatan waktu dalam bulan,3066718,48,0.002
6,2023-01,keunikan: baris tidak duplikat,3066766,0,0.000
7,2023-01,validitas: PULocationID dikenal,3066766,0,0.000
8,2023-12,kelengkapan: passenger_count ada,3196564,180003,5.331
9,2023-12,validitas: jarak 0-100 mil,3303169,73398,2.174


Langkah ini untuk menangani nilai yang hilang dan data duplikat pada dataset Januari dan Desember 2023. Nilai kosong pada passenger_count dibandingkan menggunakan beberapa strategi imputasi, kemudian dipilih pengisian berdasarkan median per jam dan median keseluruhan, sedangkan data duplikat diidentifikasi menggunakan duplikasi penuh dan kunci perjalanan sebelum dilakukan penghapusan.

In [9]:
KOL = "passenger_count"

def proses_missing_duplicate(trip, nama_bulan):
    print(f"PROSES MISSING VALUE & DUPLIKAT - {nama_bulan}")

    asli = trip[KOL]

    print(
        "persen hilang:",
        round(100 * asli.isna().mean(), 3)
    )

    trip["jam"] = trip["tpep_pickup_datetime"].dt.hour

    strategi = {
        "1_dibuang": asli.dropna(),

        "2_isi_median": asli.fillna(
            asli.median()
        ),

        "3_isi_modus": asli.fillna(
            asli.mode().iloc[0]
        ),

        "4_median_perjam": asli.fillna(
            trip.groupby("jam")[KOL].transform("median")
        ),
    }

    banding = pd.DataFrame([
        {
            "strategi": nama,
            "n": len(s),
            "mean": round(s.mean(), 4),
            "median": s.median(),
            "std": round(s.std(), 4)
        }
        for nama, s in strategi.items()
    ])

    print("\nPerbandingan strategi missing value:")
    display(banding)

    trip[KOL + "_hilang"] = (
        trip[KOL].isna().astype("int8")
    )

    trip[KOL] = trip[KOL].fillna(
        trip.groupby("jam")[KOL].transform("median")
    )

    trip[KOL] = trip[KOL].fillna(
        trip[KOL].median()
    )

    print(
        "sisa missing passenger_count:",
        trip[KOL].isna().sum()
    )

    # Duplikat: periksa dua tingkat
    dup_penuh = trip.duplicated().sum()

    KUNCI = [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "total_amount"
    ]

    dup_kunci = trip.duplicated(
        subset=KUNCI
    ).sum()

    print(
        f"duplikat penuh: {dup_penuh:,} | "
        f"duplikat pada kunci: {dup_kunci:,}"
    )

    sebelum = len(trip)

    trip = trip.drop_duplicates(
        subset=KUNCI,
        keep="first"
    ).reset_index(drop=True)

    print(
        f"{sebelum:,} -> {len(trip):,} baris"
    )

    return trip

# Proses Januari 2023
trip_jan = proses_missing_duplicate(
    trip_jan,
    "Januari 2023"
)

# Proses Desember 2023
trip_des = proses_missing_duplicate(
    trip_des,
    "Desember 2023"
)

PROSES MISSING VALUE & DUPLIKAT - Januari 2023
persen hilang: 0.0

Perbandingan strategi missing value:


,strategi,n,mean,median,std
0,1_dibuang,3066765,1.3541,1.0,0.8873
1,2_isi_median,3066765,1.3541,1.0,0.8873
2,3_isi_modus,3066765,1.3541,1.0,0.8873
3,4_median_perjam,3066765,1.3541,1.0,0.8873


sisa missing passenger_count: 0
duplikat penuh: 0 | duplikat pada kunci: 0
3,066,765 -> 3,066,765 baris
PROSES MISSING VALUE & DUPLIKAT - Desember 2023
persen hilang: 0.0

Perbandingan strategi missing value:


,strategi,n,mean,median,std
0,1_dibuang,3376566,1.3864,1.0,0.891
1,2_isi_median,3376566,1.3864,1.0,0.891
2,3_isi_modus,3376566,1.3864,1.0,0.891
3,4_median_perjam,3376566,1.3864,1.0,0.891


sisa missing passenger_count: 0
duplikat penuh: 0 | duplikat pada kunci: 0
3,376,566 -> 3,376,566 baris


Langkah ini untuk melakukan pemeriksaan outlier, penyaringan data berdasarkan aturan validitas, serta integrasi dengan data zona taksi, referensi pembayaran, data cuaca, dan data hari libur. Data yang tidak memenuhi batas jarak, durasi, dan nilai transaksi dibuang, sedangkan data dari sumber eksternal digabungkan berdasarkan lokasi, waktu, dan tanggal perjalanan.

In [12]:
def batas_iqr(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


def ringkas_outlier(df, kolom):
    hasil = []

    for kol in kolom:
        s = df[kol].dropna()

        lo, hi = batas_iqr(s)

        z = (s - s.mean()) / s.std()

        hasil.append({
            "kolom": kol,
            "batas_bawah": round(lo, 2),
            "batas_atas": round(hi, 2),
            "outlier_iqr": int(
                ((s < lo) | (s > hi)).sum()
            ),
            "outlier_z3": int(
                (z.abs() > 3).sum()
            ),
            "p99": round(
                s.quantile(0.99), 2
            ),
        })

    return pd.DataFrame(hasil)

def proses_bersih(trip, nama_bulan):
    print(f"OUTLIER, FILTERING & JOIN - {nama_bulan}")

    print("\nRingkasan outlier:")

    display(
        ringkas_outlier(
            trip,
            [
                "trip_distance",
                "durasi_menit",
                "total_amount"
            ]
        )
    )

    layak = (
        trip["trip_distance"].between(0.01, 100)
        & trip["durasi_menit"].between(1, 180)
        & (trip["total_amount"] > 0)
    )

    bersih = trip.loc[layak].copy()

    # Tandai, jangan buang outlier bisa jadi memang kenyataan bisnis
    lo, hi = batas_iqr(
        bersih["total_amount"]
    )

    bersih["tarif_ekstrem"] = (
        bersih["total_amount"] > hi
    ).astype("int8")

    # Winsorizing hanya untuk kolom turunan yang akan dipakai model
    batas_atas = bersih[
        "trip_distance"
    ].quantile(0.995)

    bersih["trip_distance_capped"] = (
        bersih["trip_distance"].clip(
            upper=batas_atas
        )
    )

    print(
        f"{len(trip):,} -> {len(bersih):,} baris "
        f"({100*(1-len(bersih)/len(trip)):.2f}% dibuang)"
    )

    print(
        "kunci zona unik?",
        zona["LocationID"].is_unique
    )

    n0 = len(bersih)

    bersih = bersih.merge(
        zona[
            ["LocationID", "Borough", "Zone"]
        ].rename(
            columns={
                "Borough": "borough_naik",
                "Zone": "zona_naik"
            }
        ),
        left_on="PULocationID",
        right_on="LocationID",
        how="left",
        validate="many_to_one"
    )

    bersih = bersih.drop(
        columns="LocationID"
    )

    print(
        f"baris {n0:,} -> {len(bersih):,} "
        f"(harus sama)"
    )

    print(
        "zona tak dikenal:",
        bersih["zona_naik"].isna().sum()
    )

    bersih = bersih.merge(
        tarif_referensi[
            ["payment_type", "nama_pembayaran"]
        ],
        on="payment_type",
        how="left",
        validate="many_to_one"
    )

    bersih["jam_mulai"] = (
        bersih[
            "tpep_pickup_datetime"
        ].dt.floor("h")
    )

    n0 = len(bersih)

    bersih = bersih.merge(
        cuaca,
        on="jam_mulai",
        how="left",
        validate="many_to_one"
    )

    print(
        f"baris {n0:,} -> {len(bersih):,}"
    )

    print(
        "tanpa data cuaca:",
        bersih["suhu_c"].isna().sum()
    )

    bersih["tanggal"] = (
        bersih[
            "tpep_pickup_datetime"
        ].dt.normalize()
    )

    # Hanya gunakan kolom yang diperlukan agar tidak menambah kolom yang tidak diperlukan
    kolom_libur = [
        "tanggal",
        "nama_libur_lokal",
        "nama_libur",
        "libur_nasional",
        "jenis_libur"
    ]

    bersih = bersih.merge(
        libur[kolom_libur],
        on="tanggal",
        how="left",
        validate="many_to_one"
    )

    # Tidak ada hari libur berarti bukan hari libur
    bersih["libur_nasional"] = (
        bersih["libur_nasional"]
        .fillna(False)
        .astype("int8")
    )

    bersih["nama_libur_lokal"] = (
        bersih["nama_libur_lokal"]
        .fillna("Bukan hari libur")
    )

    bersih["nama_libur"] = (
        bersih["nama_libur"]
        .fillna("Bukan hari libur")
    )

    print(
        "perjalanan pada hari libur:",
        int(bersih["libur_nasional"].sum())
    )

    return bersih

bersih_jan = proses_bersih(
    trip_jan,
    "Januari 2023"
)

bersih_des = proses_bersih(
    trip_des,
    "Desember 2023"
)

print("\nUkuran data setelah proses Cell 7:")
print(
    "Januari  :", f"{len(bersih_jan):,}", "baris"
)
print(
    "Desember :", f"{len(bersih_des):,}", "baris"
)

OUTLIER, FILTERING & JOIN - Januari 2023

Ringkasan outlier:


,kolom,batas_bawah,batas_atas,outlier_iqr,outlier_z3,p99
0,trip_distance,-2.35,6.74,390244,67,20.06
1,durasi_menit,-9.66,35.08,170615,3175,57.25
2,total_amount,-4.55,48.65,371616,87248,101.94


3,066,765 -> 2,986,910 baris (2.60% dibuang)
kunci zona unik? True
baris 2,986,910 -> 2,986,910 (harus sama)
zona tak dikenal: 37609
baris 2,986,910 -> 2,986,910
tanpa data cuaca: 37


/tmp/ipykernel_8621/2367041516.py:177: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


perjalanan pada hari libur: 141392
OUTLIER, FILTERING & JOIN - Desember 2023

Ringkasan outlier:


,kolom,batas_bawah,batas_atas,outlier_iqr,outlier_z3,p99
0,trip_distance,-2.38,6.62,431410,35,20.03
1,durasi_menit,-13.43,43.31,186364,3225,72.70
2,total_amount,-7.44,54.96,395752,58546,104.88


3,376,566 -> 3,254,175 baris (3.62% dibuang)
kunci zona unik? True
baris 3,254,175 -> 3,254,175 (harus sama)
zona tak dikenal: 12244
baris 3,254,175 -> 3,254,175
tanpa data cuaca: 37


/tmp/ipykernel_8621/2367041516.py:177: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


perjalanan pada hari libur: 43311

Ukuran data setelah proses Cell 7:
Januari  : 2,986,910 baris
Desember : 3,254,175 baris


Langkah ini untuk melakukan feature engineering pada dataset yang telah dibersihkan dengan membentuk fitur hari dalam minggu, akhir pekan, kecepatan perjalanan, tarif per mil, indikator hujan, tahun, dan bulan. Selain itu, data Januari dan Desember digabungkan untuk menyiapkan contoh data yang digunakan dalam proses pemisahan data latih dan uji serta transformasi fitur numerik dan kategorikal.

In [14]:
def feature_engineering(bersih, nama_bulan):
    print(f"FEATURE ENGINEERING - {nama_bulan}")

    bersih["hari_minggu"] = (
        bersih["tpep_pickup_datetime"].dt.dayofweek
    )

    bersih["akhir_pekan"] = (
        bersih["hari_minggu"] >= 5
    ).astype("int8")

    bersih["kecepatan_mph"] = (
        bersih["trip_distance"]
        / (bersih["durasi_menit"] / 60)
    ).round(2)

    bersih["tarif_per_mil"] = (
        bersih["total_amount"]
        / bersih["trip_distance"]
    ).round(3)

    bersih["hujan"] = (
        bersih["hujan_mm"].fillna(0) > 0.1
    ).astype("int8")

    # Penanda bulan untuk output partition
    bersih["tahun"] = (
        bersih["tpep_pickup_datetime"].dt.year
    ).astype("int16")

    bersih["bulan"] = (
        bersih["tpep_pickup_datetime"].dt.month
    ).astype("int8")

    print(
        "Jumlah baris:",
        f"{len(bersih):,}"
    )

    print(
        "Rentang tanggal:",
        bersih["tpep_pickup_datetime"].min(),
        "sampai",
        bersih["tpep_pickup_datetime"].max()
    )

    return bersih

bersih_jan = feature_engineering(
    bersih_jan,
    "Januari 2023"
)

bersih_des = feature_engineering(
    bersih_des,
    "Desember 2023"
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

FITUR_NUM = [
    "trip_distance_capped",
    "durasi_menit",
    "suhu_c"
]

FITUR_KAT = [
    "nama_pembayaran",
    "borough_naik"
]

contoh = pd.concat(
    [bersih_jan, bersih_des],
    ignore_index=True
)

contoh = contoh.dropna(
    subset=FITUR_NUM + FITUR_KAT
).sample(
    n=min(200_000, len(contoh)),
    random_state=42
)

latih, uji = train_test_split(
    contoh,
    test_size=0.2,
    random_state=42
)

skala = StandardScaler().fit(
    latih[FITUR_NUM]
)

latih_num = skala.transform(
    latih[FITUR_NUM]
)

uji_num = skala.transform(
    uji[FITUR_NUM]
)

print(
    "mean latih (harus ~0):",
    latih_num.mean(axis=0).round(3)
)

print(
    "mean uji (tidak harus 0):",
    uji_num.mean(axis=0).round(3)
)

enc = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

enc.fit(
    latih[FITUR_KAT]
)

latih_kat = enc.transform(
    latih[FITUR_KAT]
)

print(
    "jumlah kolom hasil one-hot:",
    latih_kat.shape[1]
)

print(
    "nama kolom:",
    enc.get_feature_names_out()[:6],
    "..."
)

FEATURE ENGINEERING - Januari 2023
Jumlah baris: 2,986,910
Rentang tanggal: 2022-10-25 00:42:10 sampai 2023-02-01 00:56:53
FEATURE ENGINEERING - Desember 2023
Jumlah baris: 3,254,175
Rentang tanggal: 2023-11-30 22:54:23 sampai 2024-01-03 19:42:57
mean latih (harus ~0): [-0.  0.  0.]
mean uji (tidak harus 0): [-0.002  0.005 -0.003]
jumlah kolom hasil one-hot: 11
nama kolom: ['nama_pembayaran_Gratis' 'nama_pembayaran_Kartu kredit'
 'nama_pembayaran_Sengketa' 'nama_pembayaran_Tunai' 'borough_naik_Bronx'
 'borough_naik_Brooklyn'] ...


Langkah ini untuk menjalankan pipeline final secara konsisten pada data Januari dan Desember 2023, mulai dari pembersihan, deduplikasi, penyaringan validitas, penggabungan berbagai sumber data, hingga pembentukan fitur akhir. Hasil kedua bulan digabungkan dalam satu direktori dengan partisi tahun, bulan, dan borough, kemudian dilakukan pemeriksaan duplikat antar-bulan serta penyimpanan laporan kualitas, data lineage, dan dataset dalam format ZIP.

In [16]:
KUNCI = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
         "PULocationID", "DOLocationID", "total_amount"]

def pipeline(path_trip, nama_bulan):
    df = pd.read_parquet(path_trip, columns=KOLOM)

    df["durasi_menit"] = (
        (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"])
        .dt.total_seconds() / 60
    )

    df["jam"] = df["tpep_pickup_datetime"].dt.hour
    df["passenger_count_hilang"] = df["passenger_count"].isna().astype("int8")
    df["passenger_count"] = df["passenger_count"].fillna(
        df.groupby("jam")["passenger_count"].transform("median")
    )
    df["passenger_count"] = df["passenger_count"].fillna(
        df["passenger_count"].median()
    )

    sebelum = len(df)
    df = df.drop_duplicates(subset=KUNCI).reset_index(drop=True)
    lineage.append({
        "sumber": f"deduplikasi_{nama_bulan}",
        "asal": "pipeline",
        "diambil_pada": pd.Timestamp.now("UTC").isoformat(),
        "jumlah_baris": len(df),
        "keterangan": f"{sebelum-len(df):,} baris duplikat dibuang"
    })

    sebelum = len(df)
    df = df[
        df["trip_distance"].between(0.01, 100)
        & df["durasi_menit"].between(1, 180)
        & (df["total_amount"] > 0)
    ].copy()

    lineage.append({
        "sumber": f"filter_validitas_{nama_bulan}",
        "asal": "pipeline",
        "diambil_pada": pd.Timestamp.now("UTC").isoformat(),
        "jumlah_baris": len(df),
        "keterangan": f"{sebelum-len(df):,} baris tidak valid dibuang"
    })

    df = df.merge(
        zona[["LocationID", "Borough", "Zone"]].rename(
            columns={"Borough": "borough_naik", "Zone": "zona_naik"}
        ),
        left_on="PULocationID", right_on="LocationID",
        how="left", validate="many_to_one"
    ).drop(columns="LocationID")

    df = df.merge(
        tarif_referensi[["payment_type", "nama_pembayaran"]],
        on="payment_type", how="left", validate="many_to_one"
    )

    df["jam_mulai"] = df["tpep_pickup_datetime"].dt.floor("h")
    df = df.merge(
        cuaca, on="jam_mulai", how="left", validate="many_to_one"
    )

    df["tanggal"] = df["tpep_pickup_datetime"].dt.normalize()
    df = df.merge(
        libur[[
            "tanggal", "nama_libur_lokal", "nama_libur",
            "libur_nasional", "jenis_libur"
        ]],
        on="tanggal", how="left", validate="many_to_one"
    )

    df["libur_nasional"] = (
        df["libur_nasional"]
        .fillna(False)
        .astype("int8")
    )

    for kol in ["nama_libur_lokal", "nama_libur"]:
        df[kol] = df[kol].fillna("Bukan hari libur")

    df["jenis_libur"] = df["jenis_libur"].apply(
        lambda x: "Bukan hari libur"
        if x is None or (isinstance(x, float) and pd.isna(x))
        else ", ".join(x) if isinstance(x, (list, tuple, np.ndarray))
        else str(x)
    )

    # Feature engineering
    df["hari_minggu"] = df["tpep_pickup_datetime"].dt.dayofweek
    df["akhir_pekan"] = (df["hari_minggu"] >= 5).astype("int8")
    df["kecepatan_mph"] = df["trip_distance"] / (df["durasi_menit"] / 60)
    df["tarif_per_mil"] = df["total_amount"] / df["trip_distance"]
    df["hujan"] = (df["hujan_mm"].fillna(0) > 0.1).astype("int8")
    df["tahun"] = df["tpep_pickup_datetime"].dt.year.astype("int16")
    df["bulan"] = df["tpep_pickup_datetime"].dt.month.astype("int8")

    lo, hi = batas_iqr(df["total_amount"])
    df["tarif_ekstrem"] = (df["total_amount"] > hi).astype("int8")

    batas_atas = df["trip_distance"].quantile(0.995)
    df["trip_distance_capped"] = df["trip_distance"].clip(upper=batas_atas)

    print(f"{nama_bulan}: {len(df):,} baris")
    return df


hasil_jan = ukur(
    "pipeline Januari",
    lambda: pipeline(PATH_TRIP_JAN, "Januari")
)

hasil_des = ukur(
    "pipeline Desember",
    lambda: pipeline(PATH_TRIP_DES, "Desember")
)

hasil = pd.concat([hasil_jan, hasil_des], ignore_index=True)

# Cek duplikat antar bulan
duplikat_antar_bulan = hasil.duplicated(subset=KUNCI).sum()
print("Duplikat antar-bulan:", f"{duplikat_antar_bulan:,}")

# Simpan dalam satu direktori dengan partisi tahun/bulan/zona
OUT = os.path.join(DIR_KURASI, "trips_bersih")

if os.path.exists(OUT):
    shutil.rmtree(OUT)

hasil.to_parquet(
    OUT,
    engine="pyarrow",
    partition_cols=["tahun", "bulan", "borough_naik"],
    index=False
)

# Laporan kualitas dua bulan
laporan = pd.concat(
    [laporan_jan.assign(bulan="Januari"),
     laporan_des.assign(bulan="Desember")],
    ignore_index=True
)

banding_kualitas = laporan.pivot(
    index="aturan",
    columns="bulan",
    values="persen_gagal"
).reset_index()

banding_kualitas["selisih_persen"] = (
    banding_kualitas["Desember"] - banding_kualitas["Januari"]
).round(3)

display(banding_kualitas.sort_values(
    "selisih_persen", key=lambda x: x.abs(), ascending=False
))

# Simpan artefak tugas
laporan.to_csv(
    os.path.join(DIR_SIMPAN, "laporan_kualitas.csv"),
    index=False
)

pd.DataFrame(lineage).to_csv(
    os.path.join(DIR_SIMPAN, "lineage.csv"),
    index=False
)

# Zip dataset final
shutil.make_archive(
    os.path.join(DIR_SIMPAN, "trips_bersih"),
    "zip",
    OUT
)

print("\nHASIL AKHIR")
print("Total baris :", f"{len(hasil):,}")
print("Januari     :", f"{len(hasil_jan):,}")
print("Desember    :", f"{len(hasil_des):,}")
print("Duplikat    :", f"{duplikat_antar_bulan:,}")
print("Output      :", OUT)

/tmp/ipykernel_8621/288321770.py:75: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


Januari: 2,986,910 baris
[pipeline Januari] 16.25 s


/tmp/ipykernel_8621/288321770.py:75: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


Desember: 3,254,175 baris
[pipeline Desember] 17.04 s
Duplikat antar-bulan: 0


bulan,aturan,Desember,Januari,selisih_persen
0,kelengkapan: passenger_count ada,5.331,2.339,2.992
6,validitas: jarak 0-100 mil,2.174,1.498,0.676
4,konsistensi: total >= fare,1.173,0.816,0.357
7,validitas: total_amount > 0,1.193,0.840,0.353
3,konsistensi: durasi 1-180 menit,1.285,1.186,0.099
1,ketepatan waktu dalam bulan,0.001,0.002,-0.001
2,keunikan: baris tidak duplikat,0.000,0.000,0.000
5,validitas: PULocationID dikenal,0.000,0.000,0.000



HASIL AKHIR
Total baris : 6,241,085
Januari     : 2,986,910
Desember    : 3,254,175
Duplikat    : 0
Output      : /content/lapisan_terkurasi/trips_bersih


Langkah ini untuk menerapkan pengolahan data menggunakan Apache Spark sebagai bagian dari pemrosesan Big Data serta menghasilkan dataset Spark sebagai hasil pemrosesan tambahan. Pada bagian akhir dibuat kamus data yang mendokumentasikan seluruh kolom dataset final, meliputi nama kolom, tipe data, satuan, sumber data, aturan validitas, dan penanganan nilai yang hilang.

In [18]:
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, unix_timestamp

spark = (
    SparkSession.builder
    .appName("Praktikum3-BigData")
    .getOrCreate()
)

# Baca dan gabungkan dua bulan
sdf_jan = spark.read.parquet(PATH_TRIP_JAN)
sdf_des = spark.read.parquet(PATH_TRIP_DES)
sdf = sdf_jan.unionByName(sdf_des)

# Pembersihan dengan Spark
sdf = (
    sdf
    .withColumn(
        "durasi_menit",
        (
            unix_timestamp("tpep_dropoff_datetime")
            - unix_timestamp("tpep_pickup_datetime")
        ) / 60
    )
    .filter(
        col("trip_distance").between(0.01, 100)
        & col("durasi_menit").between(1, 180)
        & (col("total_amount") > 0)
    )
    .dropDuplicates(KUNCI)
)

print("Jumlah baris Spark:", f"{sdf.count():,}")

# Simpan hasil Spark
OUT_SPARK = os.path.join(DIR_KURASI, "trips_spark")

if os.path.exists(OUT_SPARK):
    shutil.rmtree(OUT_SPARK)

sdf.write.mode("overwrite").partitionBy(
    "PULocationID"
).parquet(OUT_SPARK)

print("Output Spark:", OUT_SPARK)

spark.stop()

# KAMUS DATA
kamus = {
    "tpep_pickup_datetime": ("datetime", "waktu", "TLC Yellow Taxi", "Waktu pickup valid", "Tidak diimputasi"),
    "tpep_dropoff_datetime": ("datetime", "waktu", "TLC Yellow Taxi", "Waktu dropoff valid", "Tidak diimputasi"),
    "passenger_count": ("float64", "penumpang", "TLC Yellow Taxi", "Nilai jumlah penumpang", "Median per jam lalu median keseluruhan"),
    "trip_distance": ("float64", "mil", "TLC Yellow Taxi", "0.01-100 mil", "Baris di luar batas dibuang"),
    "PULocationID": ("int64", "ID zona", "TLC Yellow Taxi", "ID zona valid", "Tidak diimputasi"),
    "DOLocationID": ("int64", "ID zona", "TLC Yellow Taxi", "ID zona tujuan", "Tidak diimputasi"),
    "payment_type": ("int64", "kode", "TLC Yellow Taxi", "Kode pembayaran", "Tidak diimputasi"),
    "fare_amount": ("float64", "USD", "TLC Yellow Taxi", "Nilai tarif", "Tidak diimputasi"),
    "tip_amount": ("float64", "USD", "TLC Yellow Taxi", "Nilai tip", "Tidak diimputasi"),
    "total_amount": ("float64", "USD", "TLC Yellow Taxi", "> 0", "Baris <= 0 dibuang"),
    "durasi_menit": ("float64", "menit", "Turunan pickup-dropoff", "1-180 menit", "Baris di luar batas dibuang"),
    "jam": ("int64", "jam", "Turunan pickup", "0-23", "Tidak diimputasi"),
    "passenger_count_hilang": ("int8", "indikator", "Turunan passenger_count", "0 atau 1", "Tidak ada missing"),
    "borough_naik": ("object", "-", "Taxi Zone Lookup", "Borough sesuai PULocationID", "Zona tidak dikenal tetap missing"),
    "zona_naik": ("object", "-", "Taxi Zone Lookup", "Nama zona sesuai PULocationID", "Zona tidak dikenal tetap missing"),
    "nama_pembayaran": ("object", "-", "Referensi tarif", "Kode pembayaran memiliki referensi", "Kode tidak dikenal menjadi missing"),
    "jam_mulai": ("datetime", "waktu", "Turunan pickup", "Awal jam", "Tidak diimputasi"),
    "suhu_c": ("float64", "°C", "Open-Meteo", "Suhu dari API cuaca", "Tidak diimputasi"),
    "hujan_mm": ("float64", "mm", "Open-Meteo", "Curah hujan", "Missing tetap missing"),
    "tanggal": ("datetime", "tanggal", "Turunan pickup", "Tanggal perjalanan", "Tidak diimputasi"),
    "nama_libur_lokal": ("object", "-", "Nager.Date API", "Nama hari libur lokal", "Bukan hari libur"),
    "nama_libur": ("object", "-", "Nager.Date API", "Nama hari libur", "Bukan hari libur"),
    "libur_nasional": ("int8", "indikator", "Nager.Date API", "0 atau 1", "Missing menjadi 0"),
    "jenis_libur": ("object", "-", "Nager.Date API", "Jenis hari libur", "Bukan hari libur"),
    "hari_minggu": ("int64", "hari", "Turunan pickup", "0=Senin sampai 6=Minggu", "Tidak diimputasi"),
    "akhir_pekan": ("int8", "indikator", "Turunan hari_minggu", "0 atau 1", "Tidak ada missing"),
    "kecepatan_mph": ("float64", "mil/jam", "Turunan jarak dan durasi", "Jarak/durasi", "Tidak diimputasi"),
    "tarif_per_mil": ("float64", "USD/mil", "Turunan tarif dan jarak", "Total tarif/jarak", "Tidak diimputasi"),
    "hujan": ("int8", "indikator", "Turunan hujan_mm", "1 jika > 0.1 mm", "Missing dianggap 0"),
    "tahun": ("int16", "tahun", "Turunan pickup", "Tahun perjalanan", "Tidak diimputasi"),
    "bulan": ("int8", "bulan", "Turunan pickup", "1-12", "Tidak diimputasi"),
    "tarif_ekstrem": ("int8", "indikator", "Turunan IQR", "1 jika tarif di atas batas IQR", "Tidak ada missing"),
    "trip_distance_capped": ("float64", "mil", "Turunan trip_distance", "Dibatasi persentil 99.5%", "Tidak diimputasi"),
}

assert set(hasil.columns) == set(kamus.keys())

isi = [
    "# Kamus Data Dataset Final",
    "",
    f"Jumlah baris: **{len(hasil):,}**",
    f"Jumlah kolom: **{len(hasil.columns)}**",
    "",
    "| Nama Kolom | Tipe | Satuan | Sumber | Aturan Validitas | Penanganan Missing |",
    "|---|---|---|---|---|---|"
]

for kol in hasil.columns:
    tipe, satuan, sumber, valid, missing = kamus[kol]
    isi.append(
        f"| `{kol}` | {tipe} | {satuan} | {sumber} | {valid} | {missing} |"
    )

PATH_KAMUS = os.path.join(DIR_SIMPAN, "kamus_data.md")

with open(PATH_KAMUS, "w", encoding="utf-8") as f:
    f.write("\n".join(isi))

print("\nKamus data berhasil dibuat:", PATH_KAMUS)

for nama in [
    "laporan_kualitas.csv",
    "lineage.csv",
    "kamus_data.md",
    "trips_bersih.zip"
]:
    print(nama, "->", os.path.exists(os.path.join(DIR_SIMPAN, nama)))

Jumlah baris Spark: 6,241,085
Output Spark: /content/lapisan_terkurasi/trips_spark

Kamus data berhasil dibuat: /content/drive/MyDrive/BigData/Praktikum3/tugas/kamus_data.md
laporan_kualitas.csv -> True
lineage.csv -> True
kamus_data.md -> True
trips_bersih.zip -> True
